# Cosmos Experimentation

In [1]:
!nvidia-smi


Fri Oct 30 17:22:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   28C    P0             42W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

Python and PyTorch versions

In [2]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
CUDA available: True
CUDA version: 13.0
GPU: NVIDIA A100-SXM4-40GB
BF16 supported: True


Install the required packages

In [3]:
!pip install -U git+https://github.com/huggingface/diffusers.git
!pip install -U transformers accelerate safetensors sentencepiece protobuf huggingface_hub imageio imageio-ffmpeg
%pip install --upgrade --no-deps torchao
!pip install cosmos_guardrail
!pip install decord

  Cloning https://github.com/huggingface/diffusers.git to /tmp/pip-req-build-n7haku1v
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/diffusers.git /tmp/pip-req-build-n7haku1v
  Resolved https://github.com/huggingface/diffusers.git to commit d8c1b13c756df0059bcdb6e477a1eafcd502098d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for diffusers: filename=diffusers-0.42.0.dev0-py3-none-any.whl size=6421665 sha256=b04d59915f14fe8c6448ca06f58af8f0739bae7b1133878d242366a076afaa06
  Stored in directory: /tmp/pip-ephem-wheel-cache-9u4veljp/wheels/16/f5/67/8b5e8d211f82c13384f5cd17d760f2df2fc61184f775119f20
Successfully built diffusers
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.42.0
    Uninstalling diffusers-0.42.0:
      Successfully uninstalled diffusers-0.42.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 18.0 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [cosmos_guardrail]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 38.3 MB/s  0:00:00


Verify the installation

In [4]:
import torch
import diffusers
import transformers
import accelerate
import huggingface_hub

print("PyTorch:", torch.__version__)
print("Diffusers:", diffusers.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

print("\nGPU:", torch.cuda.get_device_name(0))
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.11.0+cu130
Diffusers: 0.42.0.dev0
Transformers: 5.22.1
Accelerate: 1.15.0
Hugging Face Hub: 2.4.0

GPU: NVIDIA A100-SXM4-40GB
CUDA available: True


Cosmos3-Nano model verification

In [5]:
from huggingface_hub import HfApi

api = HfApi()

models = list(api.list_models(
    search="Cosmos3-Nano",
    limit=20
))

for model in models:
    print(model.id)

nvidia/Cosmos3-Nano-Sim-Auto
nvidia/Cosmos3-Nano
nvidia/Cosmos3-Nano-Transfer-Auto
nvidia/Cosmos3-Nano-Sim-Transfer
lerobot/cosmos3-nano
nvidia/Cosmos3-Nano-Sim-Bimanual
nvidia/cosmos3-nano-libero10-sdg-export-S2_s43
nvidia/Cosmos3-H-Embodied-Nano
nvidia/Cosmos3-Nano-Policy-DROID
serizard1005/Cosmos3-nano-dummy
benjiaiplayground/Cosmos3-Nano_fp8
iceDonkey/Cosmos3-Nano
Felldude/Cosmos3-Nano-FP8
Reza2kn/Cosmos3-Nano-INT4-AWQ
Reza2kn/Cosmos3-Nano-NVFP4-AWQ
Reza2kn/Cosmos3-Nano-FP8
mouyun/Cosmos3-Nano
DuoNeural/Cosmos3-Nano-Abliterated
DuoNeural/Cosmos3-Nano-GPTQ-4bit
SanDiegoDude/Cosmos3-Nano-nf4


In [6]:
import importlib.metadata as metadata
import torchao.quantization as q

print("TorchAO:", metadata.version("torchao"))
print("FqnToConfig available:", hasattr(q, "FqnToConfig"))

from diffusers import Cosmos3OmniPipeline

print("Cosmos3 Omni Pipeline imported successfully!")

TorchAO: 0.18.0
FqnToConfig available: True


Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`


Cosmos3 Omni Pipeline imported successfully!


Inspect the official Cosmos3-Nano checkpoint

In [7]:
from huggingface_hub import HfApi

api = HfApi()

repo_id = "nvidia/Cosmos3-Nano"

info = api.model_info(repo_id)

print("Model:", info.id)
print("Gated:", info.gated)

print("\nRepository files:")
for file in info.siblings:
    print(file.rfilename)

Model: nvidia/Cosmos3-Nano
Gated: False

Repository files:
.gitattributes
BIAS.md
EXPLAINABILITY.md
PRIVACY.md
README.md
SAFETY.md
assets/example_action_fd_agibotworld_4chunk_output.mp4
assets/example_action_fd_agibotworld_action_chunks.json
assets/example_action_fd_agibotworld_first_frame.png
assets/example_action_id_av_0_input.mp4
assets/example_action_id_av_0_output.json
assets/example_action_id_av_0_output.png
assets/example_action_id_av_1_input.mp4
assets/example_action_id_av_1_output.json
assets/example_action_id_av_1_output.png
assets/example_i2v_input.jpg
assets/example_i2v_output.mp4
assets/example_i2v_prompt.json
assets/example_i2vs_output.mp4
assets/example_reasoning_input.png
assets/example_reasoning_prompt.json
assets/example_t2v_diffusers_output.mp4
assets/example_t2v_output.mp4
assets/example_t2v_prompt.json
assets/example_t2v_prompt_short.txt
assets/example_t2vs_output.mp4
assets/example_t2vs_prompt.json
assets/negative_prompt.json
chat_template.json
checkpoint.json
con

Check model size before downloading

In [8]:
from huggingface_hub import HfApi

api = HfApi()

info = api.model_info(
    "nvidia/Cosmos3-Nano",
    files_metadata=True
)

total_size = 0

for file in info.siblings:
    if file.rfilename.endswith(".safetensors"):
        size_gb = (file.size or 0) / (1024**3)
        total_size += size_gb

        print(f"{file.rfilename}: {size_gb:.2f} GB")

print("-" * 50)
print(f"Total checkpoint size: {total_size:.2f} GB")

sound_tokenizer/diffusion_pytorch_model.safetensors: 1.85 GB
transformer/diffusion_pytorch_model-00001-of-00007.safetensors: 4.57 GB
transformer/diffusion_pytorch_model-00002-of-00007.safetensors: 4.66 GB
transformer/diffusion_pytorch_model-00003-of-00007.safetensors: 4.59 GB
transformer/diffusion_pytorch_model-00004-of-00007.safetensors: 4.64 GB
transformer/diffusion_pytorch_model-00005-of-00007.safetensors: 4.61 GB
transformer/diffusion_pytorch_model-00006-of-00007.safetensors: 3.97 GB
transformer/diffusion_pytorch_model-00007-of-00007.safetensors: 1.23 GB
vae/diffusion_pytorch_model.safetensors: 1.31 GB
vision_encoder/model.safetensors: 1.07 GB
--------------------------------------------------
Total checkpoint size: 32.50 GB


Check available system RAM and disk space

In [9]:
import psutil
import shutil

ram = psutil.virtual_memory()
disk = shutil.disk_usage("/content")

print("System RAM:")
print(f"Total: {ram.total / 1024**3:.2f} GiB")
print(f"Available: {ram.available / 1024**3:.2f} GiB")

print("\nDisk:")
print(f"Total: {disk.total / 1024**3:.2f} GiB")
print(f"Free: {disk.free / 1024**3:.2f} GiB")

System RAM:
Total: 83.47 GiB
Available: 80.29 GiB

Disk:
Total: 235.68 GiB
Free: 192.69 GiB


Download Cosmos3-Nano

In [10]:
from huggingface_hub import snapshot_download

model_path = snapshot_download(
    repo_id="nvidia/Cosmos3-Nano",
    local_dir="/content/Cosmos3-Nano",
    ignore_patterns=[
        "assets/*",
        "images/*",
        "*.md",
        ".gitattributes"
    ],
    max_workers=4
)

print("Model downloaded to:", model_path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 36 files:   0%|          | 0/36 [00:00<?, ?it/s]

Model downloaded to: /content/Cosmos3-Nano


Load Cosmos3-Nano with CPU offloading

In [11]:
from huggingface_hub import login
login()

In [12]:
import torch
from diffusers import Cosmos3OmniPipeline

model_path = "/content/Cosmos3-Nano"

# Clear any unused GPU cache
torch.cuda.empty_cache()

# Load model components into CPU RAM
pipe = Cosmos3OmniPipeline.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)

# Enable CPU offloading
pipe.enable_model_cpu_offload()

print("Cosmos3-Nano loaded successfully!")

print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GiB"
)

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)
The config attributes {'vision_encoder': ['transformers', 'Qwen3VLVisionModel']} were passed to Cosmos3OmniPipeline, but are not expected and will be ignored. Please verify your model_index.json configuration file.
Keyword arguments {'vision_encoder': ['transformers', 'Qwen3VLVisionModel']} are not expected by Cosmos3OmniPipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

The config attributes {'clip_output': False} were passed to AutoencoderKLWan, but are not expected and will be ignored. Please verify your config.json configuration file.
The config attributes {'dtype': 'bfloat16', 'freeze_und': False, 'initializer_range': 0.02, 'joint_attn_implementation': 'two_way', 'max_action_dim': 64, 'max_position_embeddings': 262144, 'model_type': 'qwen3_vl_text', 'position_embedding_type': 'unified_3d_mrope', 'qk_norm': False, 'qk_norm_for_diffusion': True, 'temporal_compression_factor_sound': 1, 'use_cache': True, 'use_moe': True, 'video_temporal_causal': False} were passed to Cosmos3OmniTransformer, but are not expected and will be ignored. Please verify your config.json configuration file.


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


...

tokenizer.json: downloading bytes:           |  0.00B            

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Cosmos3-Nano loaded successfully!
GPU memory allocated: 1.22 GiB


In [13]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="nvidia/Cosmos-1.0-Guardrail",
    filename="blocklist/custom/notable"
)

print("Guardrail access successful!")

Guardrail access successful!


Load the model

In [14]:
import torch
from diffusers import Cosmos3OmniPipeline

model_path = "/content/Cosmos3-Nano"

torch.cuda.empty_cache()
gen = torch.Generator(device="cuda").manual_seed(42)
pipe = Cosmos3OmniPipeline.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    enable_safety_checker=False,
)

pipe.enable_model_cpu_offload()

print("Cosmos3-Nano loaded successfully!")
print(
    f"GPU memory allocated: "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GiB"
)

The config attributes {'vision_encoder': ['transformers', 'Qwen3VLVisionModel']} were passed to Cosmos3OmniPipeline, but are not expected and will be ignored. Please verify your model_index.json configuration file.
Keyword arguments {'vision_encoder': ['transformers', 'Qwen3VLVisionModel']} are not expected by Cosmos3OmniPipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

The config attributes {'clip_output': False} were passed to AutoencoderKLWan, but are not expected and will be ignored. Please verify your config.json configuration file.
The config attributes {'dtype': 'bfloat16', 'freeze_und': False, 'initializer_range': 0.02, 'joint_attn_implementation': 'two_way', 'max_action_dim': 64, 'max_position_embeddings': 262144, 'model_type': 'qwen3_vl_text', 'position_embedding_type': 'unified_3d_mrope', 'qk_norm': False, 'qk_norm_for_diffusion': True, 'temporal_compression_factor_sound': 1, 'use_cache': True, 'use_moe': True, 'video_temporal_causal': False} were passed to Cosmos3OmniTransformer, but are not expected and will be ignored. Please verify your config.json configuration file.


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

Cosmos3-Nano loaded successfully!
GPU memory allocated: 0.00 GiB


In [15]:
# stop runall
_a

NameError: name '_a' is not defined

Run text-to-video inference

In [ ]:
from diffusers.utils import load_image

# prompt = (
#     "A view of road intersection. Two undamaged cars approach, one from the left and one going straight. For the first moment they are intact and moving normally. They meet at the center, and only then do the front ends deform on contact before both come to rest. Daytime, clear weather."
# )
# prompt = (
#     "Dashcam view from inside a blue trunk car driving forward on a highway. A silver SUV is just ahead "
#     "in the right lane, moving in the same direction, both vehicles undamaged and intact. "
#     "The silver SUV signals left, then suddenly changes lanes to the left and cuts across into "
#     "the path of the blue trunk car. The blue trunk car is moving fast. For the first moments both cars are intact"
#     "They make contact. Only on contact does the front of the blue trunk car and the rear left of the "
#     "silver SUV deform, and then both come to rest. Sunny day, clear sky, light traffic."
# )

# prompt = (
#     "Dashcam view from inside a car with a blue hood driving forward on a highway at high "
#     "speed. A silver Mercedes-Benz SUV is just ahead in the right lane, moving in the same "
#     "direction, both vehicles undamaged and intact. The silver Mercedes-Benz SUV signals "
#     "left, then suddenly changes lanes to the left and cuts across into the path of the "
#     "blue-hooded car. For the first moments both cars are intact and moving normally. The "
#     "front of the blue-hooded car strikes the rear right of the silver Mercedes-Benz SUV, "
#     "and only on contact do both deform, then come to rest. Sunny day, clear sky, light "
#     "traffic."
# )

# prompt = (
#     "Dashcam view from a car driving forward on a highway. A black SUV is just ahead in the "
#     "same lane, both vehicles fully intact and undamaged, moving normally. The camera car "
#     "gradually closes the gap and makes collides onto the rear of the black SUV. On contact, "
#     "the front of the camera car and the rear of the SUV dent and crumple, and both slow to a "
#     "stop. The vehicles stay intact until the moment they touch. Clear day."
# )
prompt = (
    "Dashcam view from a car driving forward on a highway on the left lane, "
    "The Dashcam car quickly closes the gap with the black SUV, "
    "Only then the Dashcam car collides onto the rear of the black SUV, "
    "Only on impact the front of the Dashcam car deforms and the rear of the SUV dent and crumples, "
    "Only after that both slow to a stop"
)
negative_prompt = (
    "Collision happens at the very end of the video, never in the middle"
    "The Dashcam car hits the brake or slows down, "
    "Black SUV in the front hits the brake or slows down, "
    "The Black SUV shows damage even before any impact or accident for no reason, "
    "Dash cam car and the black SUV dont make contact, "
    "Dash cam car and the black SUV never make contact, "
    "Dash cam car stops before making contact with the black SUV"
    "new car appearing, car from behind, "
    "oncoming car, side car, multiple cars, other vehicles, "
    "another car causing the collision, a different car hitting the SUV, "
    "car colliding instead of the camera car, foreground car crashing, SUV hitting another car,"
    " SUV car hitting another car infront of it, the suv disappers"
)
image = load_image("start_frame.png")   # the frame you pulled from the old video
gen = torch.Generator(device="cuda").manual_seed(42)
result = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=image,
    num_frames=133,
    height=480,
    width=640,
    generator = gen,
    num_inference_steps=40,
)

In [ ]:
from IPython.display import display, Video
from diffusers.utils import export_to_video

output_path = "/content/cosmos3_nano_output4.mp4"

export_to_video(result.video, output_path, fps=24)

display(Video(output_path, embed=True, width=640))

print(f"Video saved to: {output_path}")


## Ego Action Test

In [16]:
import torch
from diffusers.utils import load_image, export_to_video
from diffusers.pipelines.cosmos.pipeline_cosmos3_omni import CosmosActionCondition
from decord import VideoReader
from PIL import Image

DOMAIN   = "av"
RES_TIER = 480
TARGET_FPS = 24          # downsample the clip to this; export at the same rate

# ---------------------------------------------------------------------------
# load frames, downsampled to TARGET_FPS across the FULL clip
# ---------------------------------------------------------------------------
def load_video_frames(path, target_fps=TARGET_FPS):
    vr = VideoReader(path)
    src_fps = vr.get_avg_fps()
    total = len(vr)
    duration = total / src_fps
    print(f"original: {total} frames, {src_fps:.2f} fps, {duration:.1f} s")

    n_out = int(round(duration * target_fps))        # frames at target fps
    idxs = [int(round(i * (total - 1) / (n_out - 1))) for i in range(n_out)]
    frames = [Image.fromarray(vr[i].asnumpy()) for i in idxs]
    print(f"sampled: {len(frames)} frames at {target_fps} fps")
    return frames, target_fps

# ===========================================================================
# STEP 1 - inverse dynamics
# ===========================================================================
def capture_action(video_path):
    frames, src_fps = load_video_frames(video_path)
    globals()["SRC_FPS"] = src_fps
    chunk = len(frames) - 1                           # chunk matches frames - 1
    gen = torch.Generator(device="cuda").manual_seed(42)

    cond = CosmosActionCondition(
        mode="inverse_dynamics",
        chunk_size=chunk,
        domain_name=DOMAIN,
        resolution_tier=RES_TIER,
        raw_actions=None,
        video=frames,
        view_point="ego_view",
    )
    out = pipe(prompt="", action=cond, num_inference_steps=30,
               enable_safety_check=False, generator=gen)

    print("output fields:", [a for a in dir(out) if not a.startswith("_")])
    action = getattr(out, "action", None)
    if action is None:
        action = getattr(out, "raw_actions", None)
    print("captured action:", None if action is None else getattr(action, "shape", type(action)))
    return action

# ===========================================================================
# STEP 2 - forward dynamics
# ===========================================================================
def forward_dynamics(image_path, raw_actions, prompt="", negative_prompt="", out_path="fd_output.mp4"):
    image = load_image(image_path)
    gen = torch.Generator(device="cuda").manual_seed(42)
    chunk = raw_actions.shape[0]

    cond = CosmosActionCondition(
        mode="forward_dynamics",
        chunk_size=chunk,
        domain_name=DOMAIN,
        resolution_tier=RES_TIER,
        raw_actions=raw_actions,
        image=image,
        view_point="ego_view",
    )
    out = pipe(prompt=prompt, negative_prompt=negative_prompt, action=cond,
               num_inference_steps=30, enable_safety_check=False, generator=gen)

    frames = out.video
    export_to_video(frames, out_path, fps=SRC_FPS)
    print("saved:", out_path, "frames:", chunk + 1, "fps:", SRC_FPS)

In [19]:
act = capture_action("left_turn_1.mp4")
raw = act[0] if isinstance(act, list) and len(act) == 1 else act
raw_actions = torch.tensor(raw, dtype=torch.float32)
print("raw_actions shape:", raw_actions.shape)
forward_dynamics("start_frame11.png", raw_actions)
from IPython.display import display, Video
display(Video("fd_output.mp4", embed=True, width=640))

original: 662 frames, 60.00 fps, 11.0 s
sampled: 265 frames at 24 fps


  0%|          | 0/30 [00:00<?, ?it/s]

output fields: ['action', 'clear', 'copy', 'fromkeys', 'get', 'items', 'keys', 'move_to_end', 'pop', 'popitem', 'setdefault', 'sound', 'to_tuple', 'update', 'values', 'video']
captured action: <class 'list'>
raw_actions shape: torch.Size([264, 9])


/tmp/ipykernel_761/2516383985.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  raw_actions = torch.tensor(raw, dtype=torch.float32)


  0%|          | 0/30 [00:00<?, ?it/s]

saved: fd_output.mp4 frames: 265 fps: 24


## Print action vector

In [ ]:
print(raw_actions.shape)
print(raw_actions[:5])
print(raw_actions.min(dim=0).values, raw_actions.max(dim=0).values)

## Testing lateral speed

In [ ]:
import torch
T = 189
ID6 = [1.0, 0.0, 0.0, 0.0, 1.0, 0.0]   # identity rotation in this 6D layout

def make_action(fwd_per_step):
    a = torch.zeros(T, 9)
    a[:, 2] = fwd_per_step             # forward translation each step
    a[:, 3:9] = torch.tensor(ID6)      # no rotation, drive perfectly straight
    return a

# 1. constant steady forward, moderate
act_steady = make_action(0.10)

# 2. faster constant forward, should close distance and not ease off
act_fast = make_action(0.25)

# 3. all zeros, sanity check - ego should barely move
act_zero = torch.zeros(T, 9)
act_zero[:, 3:9] = torch.tensor(ID6)

In [ ]:
forward_dynamics("start_frame.png", act_steady, "fd_output_steady.mp4")
from IPython.display import display, Video
display(Video("fd_output_steady.mp4", embed=True, width=640))

In [ ]:
forward_dynamics("start_frame.png", act_fast, "fd_output_fast.mp4")
from IPython.display import display, Video
display(Video("fd_output_fast.mp4", embed=True, width=640))

In [ ]:
forward_dynamics("start_frame.png", act_zero, "fd_output_zero.mp4")
from IPython.display import display, Video
display(Video("fd_output_zero.mp4", embed=True, width=640))

In [ ]:
import cv2, numpy as np

def mean_flow(video_path):
    cap = cv2.VideoCapture(video_path)
    ok, prev = cap.read()
    prev = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)
    mags = []
    while True:
        ok, frame = cap.read()
        if not ok: break
        g = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        flow = cv2.calcOpticalFlowFarneback(prev, g, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        mags.append(np.linalg.norm(flow, axis=2).mean())
        prev = g
    cap.release()
    return float(np.mean(mags))
mean_flow("fd_output1.mp4")

In [ ]:
mean_flow("fd_output.mp4")

In [ ]:
mean_flow("fd_output_1.mp4")

## Right angular turn/

In [ ]:
import torch, math

T = 189
FWD = 1.0

def yaw_rotation_6d(angle_rad):
    # proper rotation matrix for yaw, then take first two columns (Zhou 6D)
    c, s = math.cos(angle_rad), math.sin(angle_rad)
    # yaw in the ground plane; build full R then drop 3rd column
    R = torch.tensor([
        [ c, 0.0, s],
        [0.0, 1.0, 0.0],
        [-s, 0.0, c],
    ])
    a1 = R[:, 0]   # first column
    a2 = R[:, 1]   # second column
    return torch.cat([a1, a2])   # 6 values

def steer(total, fwd, max_angle_deg, start, end):
    a = torch.zeros(total, 9)
    a[:, 2] = fwd
    # identity default
    a[:, 3:9] = yaw_rotation_6d(0.0)
    # ramp heading to max over the turn window, hold after
    for t in range(total):
        if t < start:
            ang = 0.0
        elif t < end:
            frac = (t - start) / (end - start)
            ang = math.radians(max_angle_deg) * frac
        else:
            ang = math.radians(max_angle_deg)
        a[t, 3:9] = yaw_rotation_6d(ang)
    return a

act = steer(T, FWD, max_angle_deg=25, start=30, end=110)
# run with fixed seed, empty prompt
forward_dynamics("start_frame.png", act, "fd_output_left.mp4")
from IPython.display import display, Video
display(Video("fd_output_left.mp4", embed=True, width=640))


# Controlled lane change

In [ ]:
import torch, math

T = 189
FWD = 1.0

def yaw_rotation_6d(angle_rad):
    # proper rotation matrix for yaw, then take first two columns (Zhou 6D)
    c, s = math.cos(angle_rad), math.sin(angle_rad)
    # yaw in the ground plane; build full R then drop 3rd column
    R = torch.tensor([
        [ c, 0.0, s],
        [0.0, 1.0, 0.0],
        [-s, 0.0, c],
    ])
    a1 = R[:, 0]   # first column
    a2 = R[:, 1]   # second column
    return torch.cat([a1, a2])   # 6 values

def steer(total, fwd, max_angle_deg, start, end):
    a = torch.zeros(total, 9)
    a[:, 2] = fwd
    # identity default
    a[:, 3:9] = yaw_rotation_6d(0.0)
    # ramp heading to max over the turn window, hold after
    for t in range(total):
        if t < start:
            ang = 0.0
        elif t < end:
            frac = (t - start) / (end - start)
            ang = math.radians(max_angle_deg) * frac
        else:
            ang = math.radians(max_angle_deg)
        a[t, 3:9] = yaw_rotation_6d(ang)
    return a

act = steer(T, FWD, max_angle_deg=-25, start=30, end=110)
# run with fixed seed, empty prompt
forward_dynamics("start_frame.png", act, "fd_output_left.mp4")
from IPython.display import display, Video
display(Video("fd_output_left.mp4", embed=True, width=640))


## Switches left and turns right ang -25

In [ ]:
import torch, math

T = 189
FWD = 1.0


def yaw_rotation_6d(angle_rad):
    c, s = math.cos(angle_rad), math.sin(angle_rad)
    # swap the sine sign placement to flip yaw handedness
    R = torch.tensor([
        [ c, 0.0, -s],
        [0.0, 1.0, 0.0],
        [ s, 0.0,  c],
    ])
    return torch.cat([R[:, 0], R[:, 1]])

def steer(total, fwd, max_angle_deg, start, end):
    a = torch.zeros(total, 9)
    a[:, 2] = fwd
    # identity default
    a[:, 3:9] = yaw_rotation_6d(0.0)
    # ramp heading to max over the turn window, hold after
    for t in range(total):
        if t < start:
            ang = 0.0
        elif t < end:
            frac = (t - start) / (end - start)
            ang = math.radians(max_angle_deg) * frac
        else:
            ang = math.radians(max_angle_deg)
        a[t, 3:9] = yaw_rotation_6d(ang)
    return a

act = steer(T, FWD, max_angle_deg=-25, start=30, end=110)
# run with fixed seed, empty prompt
forward_dynamics("start_frame.png", act, "fd_output_left.mp4")
from IPython.display import display, Video
display(Video("fd_output_left.mp4", embed=True, width=640))


## Turn right ang -10

In [ ]:

act = steer(T, FWD, max_angle_deg=-10, start=30, end=110)
# run with fixed seed, empty prompt
forward_dynamics("start_frame.png", act, "fd_output_left.mp4")
from IPython.display import display, Video
display(Video("fd_output_left.mp4", embed=True, width=640))


##

## Drives straight

In [ ]:

act = steer(T, FWD, max_angle_deg=0, start=30, end=110)
# run with fixed seed, empty prompt
forward_dynamics("start_frame.png", act, "fd_output_left.mp4")
from IPython.display import display, Video
display(Video("fd_output_left.mp4", embed=True, width=640))


## Control turn to left

In [ ]:
act = steer(T, FWD, max_angle_deg=10, start=30, end=110)
# run with fixed seed, empty prompt
forward_dynamics("start_frame.png", act, "fd_output_left.mp4")
from IPython.display import display, Video
display(Video("fd_output_left.mp4", embed=True, width=640))


## Expected Turn left sharper 25 but just swicthes lanes

In [ ]:

act = steer(T, FWD, max_angle_deg=15, start=30, end=110)
# run with fixed seed, empty prompt
forward_dynamics("start_frame.png", act, "fd_output_left.mp4")
from IPython.display import display, Video
display(Video("fd_output_left.mp4", embed=True, width=640))


In [ ]:
import matplotlib.pyplot as plt
import torch

# raw_actions: the captured [189, 9] tensor
ra = act if torch.is_tensor(act) else torch.tensor(act, dtype=torch.float32)

t = range(ra.shape[0])
labels = ["a1x (col3)", "a1y (col4)", "a1z col5 yaw", "a2x (col6)", "a2y (col7)", "a2z (col8)"]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# rotation columns 3-8
for i, lab in enumerate(labels):
    ax1.plot(t, ra[:, 3 + i], label=lab)
ax1.axhline(1.0, color="gray", lw=0.5, ls="--")
ax1.axhline(0.0, color="gray", lw=0.5, ls="--")
ax1.set_ylabel("rotation value")
ax1.set_title("Rotation columns (6D) over time")
ax1.legend(loc="upper right", fontsize=8)

# forward column for context
ax2.plot(t, ra[:, 2], color="black", label="forward (col2)")
ax2.set_ylabel("forward")
ax2.set_xlabel("frame")
ax2.legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.savefig("action_plot.png", dpi=100)
plt.show()

## Steering with steer rate instead (constant rate every frame)

In [ ]:
import torch, math

T = 189
FWD = 1.0

def yaw_rotation_6d(angle_rad):
    c, s = math.cos(angle_rad), math.sin(angle_rad)
    # yaw about vertical (y) axis, OpenCV: x right, y down, z forward
    # rotate BOTH basis vectors in the x-z plane
    R = torch.tensor([
        [ c, 0.0, s],
        [0.0, 1.0, 0.0],
        [-s, 0.0, c],
    ])
    a1 = R[:, 0]   # [c, 0, -s]   -> x axis rotates
    a2 = R[:, 1]   # [0, 1, 0]    -> this is the problem, y column doesn't move under yaw
    return torch.cat([a1, a2])

def steer_rate(total, fwd, deg_per_frame, start, end):
    a = torch.zeros(total, 9)
    a[:, 2] = fwd
    for t in range(total):
        rate = deg_per_frame if start <= t < end else 0.0
        a[t, 3:9] = yaw_rotation_6d(math.radians(rate))
    return a

# turn during frames 40-90, hold straight after
act = steer_rate(T, FWD, deg_per_frame=18.0, start=0, end=40)

prompt = "A wide empty road, clear day."

gen = torch.Generator(device="cuda").manual_seed(42)
forward_dynamics("start_frame_2.png", act, prompt, "fd_output_cus.mp4")
from IPython.display import display, Video
display(Video("fd_output_cus.mp4", embed=True, width=640))

In [ ]:
import torch, math

ra = act if torch.is_tensor(act) else torch.tensor(act, dtype=torch.float32)

def yaw_from_6d(row):
    a1 = row[:3]
    return math.degrees(math.atan2(a1[2].item(), a1[0].item()))

print("frame | fwd | yaw_deg | raw_rot[3:9]")
for t in range(0, ra.shape[0], 5):
    row = ra[t]
    fwd = row[2].item()
    yaw = yaw_from_6d(row)
    rot = [round(x, 3) for x in row[3:9].tolist()]
    print(f"{t:5d} | {fwd:5.2f} | {yaw:7.2f} | {rot}")

## Testing with JSON Prompt

In [ ]:
import json
crash_prompt = {
    "subjects": [
        {
            "description": "A dark gray sedan directly ahead in the middle lane, in normal condition before the crash",
            "appearance_details": "An ordinary dark gray four-door sedan, intact and undamaged before impact",
            "relationship": "The target vehicle the dashcam car collides with; the dashcam car veers into its rear",
            "location": "Center of frame, in the middle lane, just ahead of the camera",
            "relative_size": "Medium at first, filling much of the frame at the impact",
            "orientation": "Facing away from the camera, same direction of travel",
            "pose": "Vehicle in motion",
            "action": "Drives forward in the middle lane at steady speed, stays in its lane",
            "state_changes": "Grows in the frame as the camera closes in, its rear end crumples at the impact at 0:04, then it is shoved forward and comes to rest",
            "number_of_subjects": 1, "number_of_arms": 0, "number_of_legs": 0
        },
        {
            "description": "The hood of the dashcam car, a sedan, visible at the bottom of the frame",
            "appearance_details": "Partial view of the hood through the windshield, typical dashcam framing",
            "relationship": "The car the camera is mounted in; it moves forward and to the right into the dark gray sedan",
            "location": "Bottom edge of frame (partial), starting in the left lane",
            "relative_size": "Small within frame",
            "orientation": "Facing forward (camera perspective), drifting right toward the middle lane",
            "pose": "Vehicle in motion",
            "action": "Accelerates forward and moves right from the left lane into the middle lane, closing on the dark gray sedan ahead",
            "state_changes": "The hood buckles and crumples at the impact at 0:04; the car stops with its front end crushed in",
            "number_of_subjects": 1, "number_of_arms": 0, "number_of_legs": 0
        }
    ],
    "background_setting": "A multi-lane road approaching a four-way intersection, daytime, dry asphalt, ordinary United States roadside surroundings. Both vehicles intact and moving normally before the crash.",
    "lighting": {
        "conditions": "Daytime, clear sky, bright even light",
        "direction": "Overhead daylight",
        "shadows": "Short shadows under the vehicles",
        "illumination_effect": "Debris and glass catch the light at the moment of impact"
    },
    "aesthetics": {
        "composition": "Dashcam view of the road ahead; the view drifts right as the camera car moves toward the middle lane and the dark gray sedan",
        "color_scheme": "Gray asphalt, dark gray target sedan, blue sky, white lane lines",
        "mood_atmosphere": "An ordinary drive that becomes a sudden rear-end collision",
        "patterns": "Painted lane lines on the road"
    },
   "cinematography": {
    "camera_motion": "The camera is rigidly fixed to the dashboard and always faces straight ahead through the windshield. It never orbits or circles any vehicle. The whole car moves forward and shifts right, so the view translates right while still facing forward, until the violent jolt at impact, then still.",
    "framing": "Wide shot of the road ahead from a windshield-mounted dashcam, fixed forward",
    "camera_angle": "Slightly below eye-level, typical dashcam mounting position, fixed to the car",
    "depth_of_field": "Deep",
    "focus": "The dark gray sedan ahead in sharp focus",
    "lens_focal_length": "Wide-angle, approximately 28mm equivalent"
},
    "style_medium": "Live-action video",
    "artistic_style": "Realistic dashcam footage, documentary-style",
    "context": "Dashcam recording of a rear-end crash: the dashcam car drifts right from the left lane into the middle lane and collides with the rear of a dark gray sedan",
    "actions": [
        {"time": "0:00-0:02", "description": "The dashcam car drives forward in the left lane. A dark gray sedan is ahead in the middle lane, both intact and moving normally."},
        {"time": "0:02-0:04", "description": "The dashcam car accelerates and drifts right out of the left lane toward the middle lane, closing fast on the dark gray sedan directly ahead."},
        {"time": "0:04-0:07", "description": "The front of the dashcam car strikes the rear of the dark gray sedan. The hood buckles, the sedan's rear crumples, glass and debris scatter, the camera lurches violently, and both cars come to rest."}
    ],
    "segments": [
        {"segment_index": 0, "time_range": "0:00-0:02", "description": "Dashcam car driving forward in the left lane, dark gray sedan ahead in the middle lane, both intact.", "key_changes": "The sedan stays ahead in the middle lane", "camera": "Forward motion with slight road vibration"},
        {"segment_index": 1, "time_range": "0:02-0:04", "description": "The dashcam car drifts right toward the middle lane, closing on the sedan.", "key_changes": "The view drifts right and the sedan grows directly ahead", "camera": "Forward motion drifting right"},
        {"segment_index": 2, "time_range": "0:04-0:07", "description": "The front of the dashcam car strikes the rear of the dark gray sedan; both crumple and come to rest.", "key_changes": "Impact: rear of sedan and front of camera car crumple, debris scatters, motion stops", "camera": "Violent jolt and shake at impact, then still"}
    ],
    "transitions": [],
    "temporal_caption": "The video opens with a daytime dashcam view from a sedan driving forward in the left lane of a multi-lane road, with a dark gray sedan ahead in the middle lane, both intact. At around 2 seconds the dashcam car accelerates and drifts right toward the middle lane, closing fast on the dark gray sedan. At 4 seconds the front of the dashcam car strikes the rear of the sedan: the hood buckles, the sedan's rear crumples, glass and debris scatter, and the camera lurches violently. In the final seconds the two wrecked cars come to rest.",
    "resolution": {"H": 480, "W": 832},
    "aspect_ratio": "16,9",
    "duration": "7s",
    "fps": 24
}

prompt_text = json.dumps(crash_prompt)


## Lane change to the right and collide with the car in front

In [ ]:
import math
T = 189
act = torch.zeros(T, 9)
act[:, 2] = 2.0                      # forward throughout
# steer right for the first stretch to change lanes toward the target
for t in range(0, 25):
    c, s = math.cos(math.radians(12)), math.sin(math.radians(12))
    R = torch.tensor([[c,0.0,s],[0.0,1.0,0.0],[-s,0.0,c]])
    act[t, 3:9] = torch.cat([R[:,0], R[:,1]])
# then straight to drive into the target
for t in range(25, T):
    act[t, 3] = 1.0
    act[t, 7] = 1.0

# prompt = prompt_text
# negative_prompt = "cars moving, cars driving, traffic flowing, car accelerating"
# gen = torch.Generator(device="cuda").manual_seed(42)
# forward_dynamics("start_frame.png", act, prompt, negative_prompt, "fd_crash_promptonly1.mp4")

In [ ]:
from IPython.display import display, Video
display(Video("fd_crash_promptonly1.mp4", embed=True, width=640))

## Prompt + iimage, no action

In [ ]:
import json

crash_prompt = {
    "subjects": [
        {
            "description": "A jeep furthest ahead in the left lane",
            "appearance_details": "An ordinary SUV-style jeep, intact, tail lights visible",
            "relationship": "The first vehicle to brake in the left-lane chain; it brakes and the pickup behind it stops in time",
            "location": "Far ahead in the left lane, in the distance",
            "relative_size": "Small, in the distance ahead",
            "orientation": "Facing away from the camera, same direction of travel",
            "pose": "Vehicle in motion, then braking",
            "action": "Drives forward, then brakes first, tail lights flaring, and slows in the left lane",
            "state_changes": "Brake lights flare at 0:01, it slows and stops in the left lane, remains undamaged",
            "number_of_subjects": 1, "number_of_arms": 0, "number_of_legs": 0
        },
        {
            "description": "A light-colored pickup truck in the left lane behind the jeep",
            "appearance_details": "An ordinary light/white pickup truck, intact before the crash, tail lights visible",
            "relationship": "Brakes in time behind the jeep, but is then struck in its right rear by the VW Polo behind it",
            "location": "Center-left of frame, ahead in the left lane, mid-distance, behind the jeep",
            "relative_size": "Medium, in the mid-distance ahead",
            "orientation": "Facing away from the camera, same direction of travel",
            "pose": "Vehicle in motion, then braking",
            "action": "Brakes in time when the jeep ahead brakes and slows safely, then is rammed from behind in its right rear by the VW Polo",
            "state_changes": "Brake lights flare at 0:02, it stops safely behind the jeep, then its right rear crumples when the Polo rams it at 0:04, and it is shoved askew across the lane",
            "number_of_subjects": 1, "number_of_arms": 0, "number_of_legs": 0
        },
        {
            "description": "A silver Volkswagen Polo hatchback in the left lane directly ahead of the camera",
            "appearance_details": "A silver VW Polo hatchback, brake lights on, intact before the crash",
            "relationship": "The last car in the left-lane chain; too fast and too close, it cannot stop in time and rams the right rear of the pickup",
            "location": "Directly ahead of the camera in the left lane, close",
            "relative_size": "Large, close ahead of the camera",
            "orientation": "Facing away from the camera, same direction of travel",
            "pose": "Vehicle in motion, braking but unable to stop",
            "action": "Driving too fast behind the pickup, brakes hard when the pickup brakes but is too close and carries forward on momentum, cannot stop, and rams straight into the right rear of the pickup truck",
            "state_changes": "Brakes at 0:03, fails to stop, its front crumples as it rams the pickup at 0:04, then it stops jammed against the pickup's right rear, blocking the left lane",
            "number_of_subjects": 1, "number_of_arms": 0, "number_of_legs": 0
        },
        {
            "description": "A white car in the right lane, ahead of the camera",
            "appearance_details": "An ordinary white car, intact, driving normally",
            "relationship": "Unaffected by the crash; drives straight on in the right lane without braking",
            "location": "Right lane, ahead of the camera",
            "relative_size": "Medium, in the mid-distance ahead on the right",
            "orientation": "Facing away from the camera, same direction of travel",
            "pose": "Vehicle in motion, steady",
            "action": "Drives straight ahead in the right lane at steady speed, does not brake, and continues past the crash unaffected",
            "state_changes": "Keeps a steady speed and drives on past, remains undamaged and does not slow",
            "number_of_subjects": 1, "number_of_arms": 0, "number_of_legs": 0
        },
        {
            "description": "The hood of the dashcam car visible at the bottom of the frame",
            "appearance_details": "Partial view of the dark hood through the windshield, typical dashcam framing",
            "relationship": "The car the camera is mounted in; follows behind the VW Polo in the left lane, a witness not involved in the crash",
            "location": "Bottom edge of frame (partial), in the left lane behind the VW Polo",
            "relative_size": "Small within frame",
            "orientation": "Facing forward (camera perspective)",
            "pose": "Vehicle in motion, slowing",
            "action": "Drives forward in the left lane behind the VW Polo, brakes and slows as the crash blocks the lane ahead",
            "state_changes": "Slows and stops behind the wreck, remains undamaged",
            "number_of_subjects": 1, "number_of_arms": 0, "number_of_legs": 0
        }
    ],
    "background_setting": "Inside a lit highway road tunnel at night, a two-lane one-way road, long curving concrete tunnel walls hard up against both sides leaving no shoulder, a continuous row of overhead ceiling lights receding into the distance, dry road surface, painted lane markings separating the left and right lanes. Enclosed tunnel interior where vehicles cannot pull aside.",
    "lighting": {
        "conditions": "Night, interior tunnel lighting, no daylight",
        "direction": "Overhead tunnel ceiling lights running down the length of the tunnel",
        "shadows": "Soft shadows under the vehicles, even artificial light along the walls",
        "illumination_effect": "Red brake lights glow in the left lane, overhead lights reflect off the tunnel walls, debris catches the light at the impact"
    },
    "aesthetics": {
        "composition": "Dashcam view down the two-lane tunnel; the left-lane chain of vehicles brakes and crashes in the mid-distance ahead while a white car in the right lane drives on unaffected and the camera trails behind in the left lane",
        "color_scheme": "Teal and gray tunnel walls, warm overhead lights, red brake lights, dark night tones",
        "mood_atmosphere": "An ordinary night drive through a tunnel that becomes a sudden chain-reaction rear-end crash in the left lane",
        "patterns": "Receding rows of ceiling lights, painted lane lines, tunnel wall panels"
    },
    "cinematography": {
        "camera_motion": "The camera is rigidly fixed to the dashboard, facing straight ahead down the tunnel, moving forward with the car and slowing as the crash unfolds ahead. It never orbits or circles any vehicle.",
        "framing": "Wide shot down the two-lane tunnel from a windshield-mounted dashcam, fixed forward",
        "camera_angle": "Slightly below eye-level, typical dashcam mounting position",
        "depth_of_field": "Deep, vehicles ahead and tunnel receding into the distance",
        "focus": "The braking vehicles ahead in the left lane in sharp focus",
        "lens_focal_length": "Wide-angle, approximately 28mm equivalent, slight barrel distortion at the edges"
    },
    "style_medium": "Live-action video",
    "artistic_style": "Realistic dashcam footage, documentary-style",
    "context": "Dashcam recording of a chain-reaction crash inside a night tunnel on a two-lane one-way road: a jeep in the left lane brakes first, the pickup behind it stops in time, but the VW Polo behind the pickup is too fast and rams the pickup's right rear, blocking the left lane, while a white car in the right lane drives on unaffected",
    "actions": [
        {"time": "0:00-0:02", "description": "The dashcam car drives forward in the left lane of the tunnel behind a silver VW Polo. Ahead in the left lane are a pickup truck and a jeep; a white car drives in the right lane. All moving normally."},
        {"time": "0:02-0:04", "description": "The jeep furthest ahead brakes first, then the pickup brakes and stops in time behind it. The VW Polo directly ahead of the camera is too fast and too close; it brakes but cannot stop, and with the tunnel walls there is no room to swerve. The white car in the right lane keeps driving straight, unaffected."},
        {"time": "0:04-0:07", "description": "The VW Polo rams straight into the right rear of the pickup truck. The pickup's right rear crumples and it is shoved askew, the Polo's front crumples, and the two come to rest jammed across the left lane, blocking it. The jeep has stopped ahead, the white car continues past in the right lane, and the dashcam car brakes and slows behind the wreck."}
    ],
    "segments": [
        {"segment_index": 0, "time_range": "0:00-0:02", "description": "Dashcam car driving forward in the left lane behind a silver VW Polo; a pickup and a jeep ahead in the left lane, a white car in the right lane, all moving normally.", "key_changes": "Normal driving through the two-lane tunnel", "camera": "Forward motion down the tunnel, fixed facing ahead"},
        {"segment_index": 1, "time_range": "0:02-0:04", "description": "The jeep brakes first, the pickup brakes and stops in time, but the VW Polo is too fast and too close to stop, with no room to swerve between the tunnel walls. The white car in the right lane drives on unaffected.", "key_changes": "Left-lane brake lights flare in sequence, the Polo closes fast on the pickup with no escape, right lane unaffected", "camera": "Forward motion, beginning to slow, fixed facing ahead"},
        {"segment_index": 2, "time_range": "0:04-0:07", "description": "The VW Polo rams the pickup's right rear; both crumple and come to rest jammed across the left lane, blocking it, while the jeep stays stopped ahead, the white car passes in the right lane, and the dashcam car slows behind.", "key_changes": "Impact ahead: right rear of pickup and front of Polo crumple, left lane blocked, right lane car drives on", "camera": "Forward motion slowing to a stop, a slight jolt as the camera car brakes, fixed facing ahead"}
    ],
    "transitions": [],
    "temporal_caption": "The video opens with a night-time dashcam view driving forward in the left lane of a two-lane one-way highway tunnel, behind a silver VW Polo, with a pickup truck and a jeep further ahead in the left lane and a white car in the right lane. At around 2 seconds the jeep brakes first and the pickup brakes and stops in time, but the VW Polo directly ahead of the camera is too fast and too close; it brakes but cannot stop and, with tunnel walls on both sides, has no room to swerve. At 4 seconds the Polo rams straight into the right rear of the pickup: the pickup's right rear crumples and it is shoved askew, the Polo's front crumples, and the two come to rest jammed across the left lane, blocking it. The jeep stays stopped ahead, the white car in the right lane drives on unaffected, and the dashcam car brakes and slows behind the wreck under the tunnel lights.",
    "resolution": {"H": 480, "W": 832},
    "aspect_ratio": "16,9",
    "duration": "7s",
    "fps": 24
}

prompt = json.dumps(crash_prompt)
negative_prompt = ("car swerving to avoid collision, vehicles avoiding each other, "
    + "car braking in time to avoid crash, no collision, cars passing through each other, "
    + "camera orbiting a vehicle, camera circling, extra invented vehicles")
gen = torch.Generator(device="cuda").manual_seed(42)
forward_dynamics("start_frame.png", None, prompt, negative_prompt, "fd_crash.mp4")
from IPython.display import display, Video
display(Video("fd_crash.mp4", embed=True, width=640))

## Load Image editing model

In [ ]:
import torch
import sys
import shutil

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
print("Free disk:", round(shutil.disk_usage("/content").free / 1024**3, 2), "GB")


In [ ]:

%pip install -U diffusers transformers accelerate huggingface_hub sentencepiece protobuf safetensors


In [ ]:
"03_approach_3.jpg"
%pip uninstall -y torchao
%pip install torchao==0.18.0


In [ ]:
from huggingface_hub import login

login()


In [ ]:

import torch
from diffusers import FluxKontextPipeline

model_id = "black-forest-labs/FLUX.1-Kontext-dev"

pipe = FluxKontextPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16
)

# Use CPU offloading to reduce GPU memory consumption
pipe.enable_model_cpu_offload()

print("FLUX Kontext loaded successfully!")


In [ ]:

import os

print(os.listdir("/content"))


In [ ]:

from PIL import Image
from IPython.display import display

image_path = "/content/03_approach_3.png"

input_image = Image.open(image_path).convert("RGB")

print("Image size:", input_image.size)
display(input_image)


In [ ]:

prompt = """
Change the background to a beautiful snowy mountain
landscape. Keep the main subject, appearance,
and composition unchanged. Maintain photorealism.
"""

result = pipe(
    image=input_image,
    prompt=prompt,
    guidance_scale=2.5,
    num_inference_steps=28,
    generator=torch.Generator("cpu").manual_seed(42)
).images[0]

display(result)

result.save("/content/kontext_edited.png")
